**Mini Project: Sentiment Assistant with BERT Fine-Tuning**


**Welcome to the final day mini project!**

In this lab you will fine-tune bert-base-uncased on movie reviews, evaluate the model, and connect the dots with real customer-support scenarios. Every section explains why we take a step, what to look for, and where an illustration could make the concept stick for your learners.

**Prerequisites & Story Setup**

**Scenario:** The support analytics team wants a dependable sentiment signal for long-form feedback so they can escalate angry customers before churn happens.

**Environment needs:** Python 3.9+, a GPU-enabled runtime (Colab, Kaggle, or a local GPU machine), and ~6 GB of free VRAM. CPU-only will work but training takes longer.

**Packages:** tensorflow, tensorflow-datasets, transformers, accelerate, and evaluate, all of which appeared earlier in the course, so you already used these tools before.

In [ ]:
# Run once in a fresh environment
%pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

In [ ]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

**Charger l'ensemble de données des critiques IMDB**

Nous utilisons IMDb car ses données sont équilibrées (25 000 avis positifs / 25 000 avis négatifs) et déjà segmentées. Les apprenants devraient le reconnaître grâce aux exemples d'analyse de sentiments précédents.

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)

In [ ]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

**Configuration du tokenizer et du pipeline de données**


BERT utilise la tokenisation WordPiece pour gérer les mots rares ou inconnus en les décomposant en sous-mots, garantissant ainsi une couverture complète et une taille de vocabulaire optimale. Il ajoute [CLS] et [SEP] tokens pour marquer les limites des phrases et permettre des tâches telles que la classification et la modélisation de paires de phrases. Les masques d'attention indiquent au modèle quels tokens sont valides et lesquels sont des tokens de remplissage, assurant ainsi que l'attention ne soit calculée que sur les entrées valides.

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

Nous importons ce tokenizer pour réutiliser le même vocabulaire que celui appris par le modèle de base en 2018.

Ensuite, convertissez les octets bruts en identifiants de jetons, masques d'attention et identifiants de segments.

In [ ]:
def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text, label):
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    return {
        "input_ids": encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2]
    }, label

def prepare_dataset(dataset):
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

N'oubliez pas que cela tf.py_function nous permet de conserver la logique de tokenisation Hugging Face au sein d'un pipeline TensorFlow, évitant ainsi de manipuler manuellement les tableaux NumPy. De plus, le brassage et le préchargement stabilisent le débit d'entraînement.



**Initialiser le modèle de réglage fin**

Nous chargeons maintenant TFBertForSequenceClassification, qui regroupe déjà l'encodeur et la tête de classification.

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

Nous importons ce modèle afin de réutiliser les 110 millions de paramètres appris sur BooksCorpus et Wikipédia. Nous effectuons un réglage fin uniquement sur quelques époques, ce qui explique les taux d'apprentissage de l'ordre de 2e-5.



**Former et superviser**
Sur un GPU T4 (celui utilisé dans Google Colab), deux époques prennent environ 15 minutes. Veuillez surveiller la précision des calculs d'entraînement et de validation.

In [ ]:
EPOCHS = 2
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=test_ds
)
print("Training completed!")
print(f"Training accuracy: {history.history['accuracy'][-1]:.4f}")
if 'val_accuracy' in history.history:
    print(f"Validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

📈 Indiquez comment le plateau de précision de la validation signale le moment d'arrêter. Encouragez également la publication de captures d'écran des courbes d'apprentissage des portefeuilles.



Évaluer sur l'ensemble de test mis de côté
Même si model.fit des indicateurs de validation sont déjà fournis, nous relançons l'évaluation sur le pipeline de test intact afin de simuler l'assurance qualité en production.

In [ ]:
eval_metrics = model.evaluate(test_ds)
print(f"Test loss: {eval_metrics[0]:.4f}")
print(f"Test accuracy: {eval_metrics[1]:.4f}")

Vous pourrez constater ici si la précision dépasse le seuil de référence de ~0,90 en classe.

À faire : Utiliser ceci pour discuter des taux d'erreur acceptables pour les équipes de support réelles.



**Créer un assistant d'inférence réutilisable**
Intégrez le tout dans une fonction afin de pouvoir coller de véritables transcriptions de justificatifs et obtenir un score instantané.

In [ ]:
def predict_sentiment(text: str):
    encoded = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="tf"
    )
    outputs = model(encoded)
    probs = tf.nn.softmax(outputs.logits, axis=-1).numpy()[0]
    pred_label = int(tf.argmax(probs).numpy())
    label = "Positive" if pred_label == 1 else "Negative"
    return label, float(probs.max())

custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")

Pour l'exemple précédent, vous devriez voir Prediction: Positive (confidence=0.5) slightly more or less.

Les scores de confiance sont essentiels pour décider s'il faut répondre automatiquement ou faire appel à un humain.



Réflexion et prochaines étapes
Pourquoi le réglage fin est important : Vous avez réutilisé un point de contrôle public pour atteindre une précision supérieure à 90 % avec un minimum de données.
Compétences transférables : Tout ce qui précède s'applique également aux tâches de classification dans les domaines des RH, du juridique ou de l'analyse de produits.
Ce que vous pouvez faire avec ceci : adaptation de domaine (collecte des e-mails de votre entreprise), points de contrôle multilingues (DistilBERT multilingue, XLM-R) et surveillance (enregistrement des dérives de données, création de tableaux de bord).

À faire : Répondre aux questions de réflexion suivantes dans les cellules Markdown de votre cahier :

Quel levier (nettoyage des données, hyperparamètres, nombre d'époques) a le plus amélioré les résultats ?
Où ajouteriez-vous des garde-fous avant de déployer ce signal de sentiment en production ?
Quels sont les acteurs qui en bénéficient le plus (responsable du support, chef de produit, responsable de la conformité) ?

### REPONSE

## Réflexion et réponses

In [ ]:
print("=== Réflexions sur le projet ===\n")

print("1. Quel levier a le plus amélioré les résultats ?")
print("   - Le nombre d'epochs: plus d'epochs permet de mieux converger")
print("   - Le learning rate: 2e-5 est optimal pour le fine-tuning BERT")
print("   - Le max_length: 256 tokens capture assez de contexte pour les avis longs\n")

print("2. Garde-fous avant de déployer en production :")
print("   - Validation croisée et monitoring des drift de données")
print("   - Calibration des probabilités de sortie")
print("   - Seuil de confiance minimum pour déclencher l'escalade")
print("   - Tests A/B avec les échanges humains\n")

print("3. Principaux bénéficiaires :")
print("   - Responsable support: priorisation des tickets clients")
print("   - Chef de produit: analyse des retours clients produits")
print("   - Responsable conformité: détection des messages sensibles")
print("   - Direction: suivi de la satisfaction client globale\n")

print("=== Métriques finales du modèle ===")
print(f"Précision sur le jeu de test: {eval_metrics[1]*100:.2f}%")